# Shapedb Search Session

This notebook prepares a PDB-derived ligand shape file and launches a shapedb search against the Enamine database.

- It saves the ligand segment `LIG` from the PDB as `.mol2`.
- Then it converts and centers that `.mol2` to `_centered.sdf` using `func/shapedb/convert_and_center_mol2.py`.
- Finally it uses the generated `.sdf` for the shapedb search.

In [ ]:
## input a pdb file with desing truncated structure
import os
from pathlib import Path
import subprocess
import sys

######### you don't need to change this part unless you know what you are doing
working_dir ="/pi/summer.thyme-umw/Ji_rosetta_discovery/"
enamine_database  = "/pi/summer.thyme-umw/enamine-REAL-2.6billion"
########
def run_cmd(cmd):
	result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
	if result.stdout:
		print(result.stdout, end="")
	if result.stderr:
		print(result.stderr, file=sys.stderr, end="")
	return result.returncode


In [3]:
import pymol
from pymol import cmd
import os
import subprocess
from pathlib import Path

pdb_path = Path(working_dir) / "input_pdb"
shapedb_output_root = Path(working_dir) / "out/shapedb"
convert_script = Path(working_dir) / "func" / "shapedb" / "convert_and_center_mol2.py"

print("working_dir:", working_dir)
print("pdb_path:", pdb_path)
print("shapedb_output_root:", shapedb_output_root)

if not pdb_path.exists():
    raise FileNotFoundError(f"PDB file not found: {pdb_path}")

for pdb_file in pdb_path.glob("*.pdb"):
    stem = pdb_file.stem
    shapedb_output = shapedb_output_root / stem
    ligand_mol2 = shapedb_output / f"{stem}.mol2"
    ligand_sdf = shapedb_output / f"{stem}_centered.sdf"

    print("Processing PDB:", pdb_file)
    shapedb_output.mkdir(parents=True, exist_ok=True)

    cmd.load(str(pdb_file), object='agonist')
    cmd.remove('chain R')
    cmd.save(str(ligand_mol2), 'segid LIG', format='mol2')
    print("Saved MOL2:", ligand_mol2)

    if not ligand_sdf.exists():
        print("Converting MOL2 to centered SDF:", ligand_mol2)
        subprocess.run(["python", str(convert_script), str(shapedb_output)], check=True)
    else:
        print("Centered SDF already exists:", ligand_sdf)

    print("Output SDF:", ligand_sdf)

working_dir: /pi/summer.thyme-umw/Ji_rosetta_discovery/
pdb_path: /pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb
shapedb_output_root: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb
Processing PDB: /pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb/orexin_state3.pdb
Saved MOL2: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/orexin_state3.mol2
Centered SDF already exists: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/orexin_state3_centered.sdf
Output SDF: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/orexin_state3_centered.sdf
Processing PDB: /pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb/orexin_state4.pdb
Saved MOL2: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state4/orexin_state4.mol2
Centered SDF already exists: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state4/orexin_state4_centered.sdf
Output SDF: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state4/or

In [ ]:
# check your output sdf, if it's good set this to True to execute the shapedb search.
run_search = True

ligand_mol2 =  "/pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/orexin_state3_centered.sdf"
shapedb_output = "/pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/"
if run_search:
    shapedb_controller = Path(working_dir) / "func" / "shapedb" / "nnsearch_controller.py"
    shapedb_cmd = ["bsub -q long -W 24:00 -R \"rusage[mem=5000]\" -J shapedb_search_ctrl" ,
        "python",
        str(shapedb_controller),
        str(ligand_mol2),
        shapedb_output,
        "/pi/summer.thyme-umw/Ji_rosetta_discovery"
    ]
    print("Executing shapedb search:", " ".join(shapedb_cmd))
    run_cmd(" ".join(shapedb_cmd))
else:
    print("Search not executed.")
    print("Set run_search = True and rerun this cell when you want to submit shapedb.")


Executing shapedb search: bsub -q long -W 24:00 -R "rusage[mem=5000]" -J shapedb_search_ctrl python /pi/summer.thyme-umw/Ji_rosetta_discovery/func/shapedb/nnsearch_controller.py /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/orexin_state3_centered.sdf /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/ /pi/summer.thyme-umw/Ji_rosetta_discovery
Job <493231> is submitted to queue <long>.


WARN: Job does not specify number of cores. Setting 'bsub -n 1' (single core).


In [14]:
num_confs_to_keep=3000000
extract_top_confs_script = Path(working_dir) / "func" / "shapedb" / "func/shapedb/get_top_x_ligands_in_whole_library_hpc.py"
extract_cmd = ["bsub -q long -W 168:00 -n 8 -R \"rusage[mem=4000]\" -J extract_top_confs" ,
        "python",
        str(extract_top_confs_script),
        str(num_confs_to_keep),
        shapedb_output
    ]

run_cmd(" ".join(extract_cmd))

Job <536613> is submitted to queue <long>.


NameError: name 'sys' is not defined